In [ ]:
import pandas as pd

# Load your SFE dataset
df = pd.read_excel("Composition_SFE_Dataset_old.xlsx")

# Show first few rows (optional)
print(df.head())

# Check if SFE column exists
print("Columns in the dataset:", df.columns.tolist())

# Compute min and max
min_sfe = df["SFE"].min()
max_sfe = df["SFE"].max()

print("\n====== SFE Range in Dataset ======")
print(f"Minimum SFE: {min_sfe:.4f}")
print(f"Maximum SFE: {max_sfe:.4f}")


         Fe        Mn        Ni        Co        Cr         V        Al  \
0  0.142857  0.142857  0.142857  0.142857  0.142857  0.142857  0.142857   
1  0.202866  0.025391  0.365391  0.296460  0.054369  0.032077  0.023446   
2  0.182712  0.006847  0.510895  0.071240  0.161001  0.038695  0.028610   
3  0.035201  0.015630  0.724721  0.141416  0.030501  0.008102  0.044428   
4  0.227773  0.014224  0.531015  0.100751  0.063310  0.012472  0.050456   

          SFE  
0   51.649535  
1   26.631791  
2   47.896873  
3  134.401773  
4   70.897057  
Columns in the dataset: ['Fe', 'Mn', 'Ni', 'Co', 'Cr', 'V', 'Al', 'SFE']

====== SFE Range in Dataset ======
Minimum SFE: -315.1430
Maximum SFE: 148.8960


In [ ]:
!pip uninstall scikit-learn -y
!pip install scikit-learn==1.5.1

Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 100.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.11 requires scikit-learn>=1.6, but you have scikit-learn 1.5.1 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.1 which is incompatible.


In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.9 MB/s eta 0:00:00


In [ ]:
import joblib

model_path = "stacked_sfe_model_PT.pkl"

clf2 = joblib.load(model_path)

In [ ]:
!pip install pymoo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.3/328.3 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 6.4 MB/s eta 0:00:00


In [ ]:
import pymoo

In [ ]:
import numpy as np
import pandas as pd
import joblib

from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.optimize import minimize


# ============================================
# 1. LOAD MODEL
# ============================================
model_path = "stacked_sfe_model_PT.pkl"
model = joblib.load(model_path)

# Exact feature order used during training
feature_order = ['Fe', 'Mn', 'Ni', 'Co', 'Cr', 'V', 'Al']

# Optional check
print("Using feature order:", feature_order)


Using feature order: ['Fe', 'Mn', 'Ni', 'Co', 'Cr', 'V', 'Al']


In [ ]:
def repair_composition(x, low=0.05, high=0.35, target_sum=1.0, max_iter=200):
    x = np.array(x, dtype=float)

    # First clip to bounds
    x = np.clip(x, low, high)

    for _ in range(max_iter):
        s = x.sum()
        diff = target_sum - s

        if abs(diff) < 1e-8:
            break

        # if need to increase total, only increase variables below upper bound
        if diff > 0:
            free = np.where(x < high - 1e-12)[0]
        else:
            free = np.where(x > low + 1e-12)[0]

        if len(free) == 0:
            break

        x[free] += diff / len(free)
        x = np.clip(x, low, high)

    return x


In [ ]:
class HEAInverseDesign(ElementwiseProblem):
    def __init__(self, model):
        super().__init__(
            n_var=7,
            n_obj=1,
            n_constr=4,
            xl=np.array([0.05] * 7),
            xu=np.array([0.35] * 7),
        )
        self.model = model

    def _evaluate(self, x, out, *args, **kwargs):
        # repair composition first
        x = repair_composition(x)

        # make dataframe in exact training order
        X_df = pd.DataFrame([x], columns=feature_order)
        X_df = X_df[feature_order]

        # predict SFE
        sfe = float(self.model.predict(X_df)[0])

        # sum of composition
        s = float(np.sum(x))

        # objective: target SFE close to 20
        out["F"] = [abs(sfe - 20.0)]

        # constraints must be <= 0 in pymoo
        g1 = 0.98 - s     # s >= 0.98
        g2 = s - 1.10     # s <= 1.10
        g3 = 15.0 - sfe   # sfe >= 15
        g4 = sfe - 25.0   # sfe <= 25

        out["G"] = [g1, g2, g3, g4]


In [ ]:
problem = HEAInverseDesign(model)

In [ ]:
# ============================================
# 5. DEFINE OPTIMIZER
# Single-objective GA is enough here
# ============================================
algorithm = GA(
    pop_size=500,
    eliminate_duplicates=True
)

In [ ]:
# 6. RUN OPTIMIZATION
# ============================================
res = minimize(
    problem,
    algorithm,
    ('n_gen', 20),
    seed=1,
    verbose=True
)


n_gen  |  n_eval  |     cv_min    |     cv_avg    |     f_avg     |     f_min    
     1 |      500 |  0.000000E+00 |  5.3784225779 |  1.9636518388 |  0.0007398861
     2 |     1000 |  0.000000E+00 |  0.0046369295 |  1.8314035182 |  0.0007398861
     3 |     1500 |  0.000000E+00 |  0.000000E+00 |  0.8772859842 |  0.0007398861
     4 |     2000 |  0.000000E+00 |  0.000000E+00 |  0.4708669970 |  0.0007398861
     5 |     2500 |  0.000000E+00 |  0.000000E+00 |  0.2908331166 |  0.0007398861
     6 |     3000 |  0.000000E+00 |  0.000000E+00 |  0.1724042058 |  0.0007398861
     7 |     3500 |  0.000000E+00 |  0.000000E+00 |  0.1114282696 |  0.0007398861
     8 |     4000 |  0.000000E+00 |  0.000000E+00 |  0.0707756000 |  0.0007398861
     9 |     4500 |  0.000000E+00 |  0.000000E+00 |  0.0501809156 |  0.0007398861
    10 |     5000 |  0.000000E+00 |  0.000000E+00 |  0.0341589630 |  0.0007398861
    11 |     5500 |  0.000000E+00 |  0.000000E+00 |  0.0267872738 |  0.0007398861
    12 |     600

In [ ]:
# ============================================
# 7. EXTRACT BEST SOLUTION
# ============================================
best_x = repair_composition(res.X)

best_df = pd.DataFrame([best_x], columns=feature_order)
best_df = best_df[feature_order]

best_sfe = float(model.predict(best_df)[0])
best_sum = float(best_df.iloc[0].sum())

print("\nBest solution:")
print(best_df)

print("\nPredicted SFE =", best_sfe)
print("Sum =", best_sum)
print("Objective |SFE-20| =", abs(best_sfe - 20.0))


# ============================================
# 8. GENERATE MANY RANDOM CANDIDATES AROUND BEST
#    AND KEEP FEASIBLE ONES


Best solution:
         Fe        Mn        Ni        Co        Cr         V        Al
0  0.070432  0.212838  0.052775  0.232982  0.109498  0.071409  0.250066

Predicted SFE = 20.000052680773585
Sum = 0.9999999999999998
Objective |SFE-20| = 5.268077358522305e-05


In [ ]:
# ============================================
# 8. GENERATE MANY RANDOM CANDIDATES AROUND BEST
#    AND KEEP FEASIBLE ONES
# ============================================
def generate_candidates_around_best(best_x, n_samples=1000, noise=0.03):
    candidates = []

    for _ in range(n_samples):
        trial = best_x + np.random.normal(0, noise, size=len(best_x))
        trial = repair_composition(trial)

        X_df = pd.DataFrame([trial], columns=feature_order)
        sfe = float(model.predict(X_df)[0])
        s = float(np.sum(trial))

        if 15.0 <= sfe <= 25.0 and 0.98 <= s <= 1.10:
            row = dict(zip(feature_order, trial))
            row["Predicted_SFE"] = sfe
            row["Sum"] = s
            row["Deviation_from_20"] = abs(sfe - 20.0)
            candidates.append(row)

    return pd.DataFrame(candidates)


df_candidates = generate_candidates_around_best(best_x, n_samples=10000, noise=0.025)

if len(df_candidates) > 0:
    df_candidates = df_candidates.sort_values(
        by=["Deviation_from_20", "Predicted_SFE"]
    ).reset_index(drop=True)

    print("\nTop feasible candidate alloys:")
    print(df_candidates.head(20))

    df_candidates.to_csv("inverse_designed_alloys_15_25_SFE.csv", index=False)
    print("\nSaved: inverse_designed_alloys_15_25_SFE.csv")
else:
    print("\nNo feasible candidate alloys found in post-search.")


Top feasible candidate alloys:
          Fe        Mn        Ni        Co        Cr         V        Al  \
0   0.059472  0.221253  0.085212  0.217958  0.101456  0.072395  0.242254   
1   0.074964  0.223264  0.050000  0.229564  0.108461  0.072624  0.241122   
2   0.067343  0.212069  0.063546  0.230565  0.115829  0.063219  0.247430   
3   0.073107  0.186348  0.050969  0.232118  0.109259  0.079547  0.268653   
4   0.069981  0.208329  0.086807  0.231354  0.114886  0.050000  0.238642   
5   0.050888  0.236517  0.052291  0.238333  0.080840  0.094640  0.246491   
6   0.086622  0.228025  0.050000  0.225808  0.133108  0.054640  0.221796   
7   0.093273  0.176238  0.062328  0.259340  0.079662  0.069449  0.259709   
8   0.050838  0.268888  0.050838  0.192055  0.097072  0.085056  0.255253   
9   0.065012  0.236423  0.090568  0.211178  0.105573  0.067029  0.224217   
10  0.066761  0.218389  0.066194  0.246737  0.085971  0.073842  0.242104   
11  0.053277  0.193828  0.063751  0.222403  0.107806  0.

In [ ]:
##########plotting#############33

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ============================================
# 8. GENERATE MANY RANDOM CANDIDATES AROUND BEST
#    AND KEEP FEASIBLE ONES
# ============================================
def generate_candidates_around_best(best_x, n_samples=1000, noise=0.03):
    candidates = []

    for i in range(n_samples):
        trial = best_x + np.random.normal(0, noise, size=len(best_x))
        trial = repair_composition(trial)

        X_df = pd.DataFrame([trial], columns=feature_order)
        sfe = float(model.predict(X_df)[0])
        s = float(np.sum(trial))

        if 15.0 <= sfe <= 25.0 and 0.98 <= s <= 1.10:
            row = dict(zip(feature_order, trial))
            row["Iteration"] = i + 1
            row["Predicted_SFE"] = sfe
            row["Sum"] = s
            row["Deviation_from_20"] = abs(sfe - 20.0)
            candidates.append(row)

    return pd.DataFrame(candidates)


df_candidates = generate_candidates_around_best(best_x, n_samples=10000, noise=0.025)

if len(df_candidates) > 0:
    print("\nTop feasible candidate alloys:")
    print(df_candidates.head(20))

    df_candidates.to_csv("inverse_designed_alloys_15_25_SFE.csv", index=False)
    print("\nSaved: inverse_designed_alloys_15_25_SFE.csv")
else:
    print("\nNo feasible candidate alloys found in post-search.")


Top feasible candidate alloys:
          Fe        Mn        Ni        Co        Cr         V        Al  \
0   0.063870  0.221041  0.050000  0.269512  0.073352  0.068561  0.253663   
1   0.071471  0.229029  0.079245  0.234231  0.087519  0.055355  0.243150   
2   0.057278  0.235831  0.057278  0.231254  0.098608  0.057278  0.262473   
3   0.076434  0.206833  0.061773  0.224545  0.114920  0.054306  0.261189   
4   0.101643  0.213689  0.064012  0.230718  0.102483  0.064012  0.223442   
5   0.050000  0.214835  0.072127  0.247508  0.107704  0.050000  0.257825   
6   0.069926  0.179428  0.050000  0.218394  0.123966  0.116459  0.241827   
7   0.050000  0.201573  0.050000  0.248346  0.099981  0.061334  0.288766   
8   0.058488  0.195692  0.058488  0.250435  0.132974  0.069723  0.234199   
9   0.053241  0.210991  0.053241  0.243665  0.116175  0.069876  0.252812   
10  0.055639  0.226258  0.055639  0.227020  0.119575  0.086181  0.229688   
11  0.078141  0.212930  0.051962  0.230668  0.116509  0.

In [ ]:
if len(df_candidates) > 0:
    plt.figure(figsize=(8,5))
    plt.plot(df_candidates["Iteration"], df_candidates["Predicted_SFE"], marker='o', linestyle='None', markersize=3)
    plt.axhline(20, linestyle='--', label='Target SFE = 20')
    plt.axhline(15, linestyle='--', label='Lower limit = 15')
    plt.axhline(25, linestyle='--', label='Upper limit = 25')

    plt.xlabel("Iteration")
    plt.ylabel("Predicted SFE")
    plt.title("Iteration vs Predicted SFE")
    plt.legend()
    plt.grid(True)
    plt.show()